# 🌲 Random Forest Model Training
**CIDRS Project** | **Date:** May 25, 2026

## Model Purpose:
- Baseline model for intrusion detection
- Fast inference time (<100ms)
- Interpretable feature importance
- Comparison benchmark for LSTM and Autoencoder

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported")

## 1. Load Preprocessed Data

In [ ]:
# Load processed data
X_train = np.load('../data/processed/X_train.npy')
X_val = np.load('../data/processed/X_val.npy')
X_test = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_val = np.load('../data/processed/y_val.npy')
y_test = np.load('../data/processed/y_test.npy')

print(f"Training set:   {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Validation set: {X_val.shape[0]} samples, {X_val.shape[1]} features")
print(f"Test set:       {X_test.shape[0]} samples, {X_test.shape[1]} features")

# Check class distribution
print(f"\nTraining - Normal: {(y_train == 0).sum()}, Attack: {(y_train == 1).sum()}")
print(f"Validation - Normal: {(y_val == 0).sum()}, Attack: {(y_val == 1).sum()}")
print(f"Test - Normal: {(y_test == 0).sum()}, Attack: {(y_test == 1).sum()}")

## 2. Train Random Forest Model

In [ ]:
# Initialize Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,      # Number of trees
    max_depth=None,        # No limit on tree depth
    random_state=42,       # Reproducibility
    n_jobs=-1,             # Use all CPU cores
    class_weight='balanced' # Handle class imbalance
)

print("Model parameters:")
print(f"  Trees: {rf_model.n_estimators}")
print(f"  Max depth: {rf_model.max_depth}")
print(f"  Class weight: {rf_model.class_weight}")

# Train the model
print("\n🚀 Training Random Forest...")
rf_model.fit(X_train, y_train)
print("✅ Training complete!")

## 3. Model Evaluation

In [ ]:
# Predictions
y_train_pred = rf_model.predict(X_train)
y_val_pred = rf_model.predict(X_val)
y_test_pred = rf_model.predict(X_test)

# Probabilities for ROC-AUC
y_train_proba = rf_model.predict_proba(X_train)[:, 1]
y_val_proba = rf_model.predict_proba(X_val)[:, 1]
y_test_proba = rf_model.predict_proba(X_test)[:, 1]

print("=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

# Training metrics
print(f"\n📊 TRAINING SET:")
print(f"  Accuracy:  {accuracy_score(y_train, y_train_pred)*100:.2f}%")
print(f"  ROC-AUC:   {roc_auc_score(y_train, y_train_proba)*100:.2f}%")

# Validation metrics
print(f"\n📊 VALIDATION SET:")
print(f"  Accuracy:  {accuracy_score(y_val, y_val_pred)*100:.2f}%")
print(f"  ROC-AUC:   {roc_auc_score(y_val, y_val_proba)*100:.2f}%")

# Test metrics
print(f"\n📊 TEST SET:")
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred)*100:.2f}%")
print(f"  Precision: {precision_score(y_test, y_test_pred)*100:.2f}%")
print(f"  Recall:    {recall_score(y_test, y_test_pred)*100:.2f}%")
print(f"  F1-Score:  {f1_score(y_test, y_test_pred)*100:.2f}%")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_test_proba)*100:.2f}%")

## 4. Classification Report

In [ ]:
print("=" * 60)
print("CLASSIFICATION REPORT (Test Set)")
print("=" * 60)
print(classification_report(y_test, y_test_pred, target_names=['Normal', 'Attack']))

## 5. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title('Random Forest - Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig('../docs/rf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Calculate rates
TN, FP, FN, TP = cm.ravel()
print(f"True Negatives (Normal correctly identified):  {TN}")
print(f"False Positives (False alarms):                {FP}")
print(f"False Negatives (Missed attacks):              {FN}")
print(f"True Positives (Attacks detected):             {TP}")
print(f"\nFalse Positive Rate: {FP/(FP+TN)*100:.2f}%")
print(f"False Negative Rate: {FN/(FN+TP)*100:.2f}%")

## 6. ROC Curve

In [ ]:
# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
roc_auc = roc_auc_score(y_test, y_test_proba)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Random Forest - ROC Curve', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/rf_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance Analysis

In [ ]:
# Get feature importances
feature_importance = rf_model.feature_importances_
feature_names = [f'F{i}' for i in range(len(feature_importance))]

# Sort by importance
sorted_idx = np.argsort(feature_importance)[::-1]
top_n = 15

# Plot top 15 features
plt.figure(figsize=(10, 8))
plt.barh(range(top_n), feature_importance[sorted_idx[:top_n]][::-1], color='steelblue')
plt.yticks(range(top_n), [f'Feature {i}' for i in sorted_idx[:top_n][::-1]])
plt.xlabel('Importance Score', fontsize=12)
plt.title(f'Top {top_n} Most Important Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top 10
print("Top 10 Most Important Features:")
for i, idx in enumerate(sorted_idx[:10]):
    print(f"  {i+1}. Feature {idx}: {feature_importance[idx]:.4f}")

## 8. Save Model

In [ ]:
# Save model
joblib.dump(rf_model, '../models/random_forest/rf_model.pkl')
print("✅ Model saved to: models/random_forest/rf_model.pkl")

# Save performance metrics
metrics = {
    'accuracy': accuracy_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred),
    'recall': recall_score(y_test, y_test_pred),
    'f1_score': f1_score(y_test, y_test_pred),
    'roc_auc': roc_auc_score(y_test, y_test_proba),
    'false_positive_rate': FP/(FP+TN),
    'false_negative_rate': FN/(FN+TP)
}

import json
with open('../models/random_forest/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("✅ Metrics saved to: models/random_forest/metrics.json")

## 9. Performance Summary

In [ ]:
print("=" * 60)
print("📋 RANDOM FOREST - FINAL RESULTS")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────────────┐
│                    MODEL PERFORMANCE                         │
├─────────────────────────────────────────────────────────────┤
│  ✅ Accuracy:      {metrics['accuracy']*100:.2f}%                                │
│  ✅ Precision:     {metrics['precision']*100:.2f}%                                │
│  ✅ Recall:        {metrics['recall']*100:.2f}%                                │
│  ✅ F1-Score:      {metrics['f1_score']*100:.2f}%                                │
│  ✅ ROC-AUC:       {metrics['roc_auc']*100:.2f}%                                │
├─────────────────────────────────────────────────────────────┤
│                    ERROR RATES                              │
├─────────────────────────────────────────────────────────────┤
│  ⚠️  False Positive Rate: {metrics['false_positive_rate']*100:.2f}% (False alarms)     │
│  ⚠️  False Negative Rate: {metrics['false_negative_rate']*100:.2f}% (Missed attacks)   │
└─────────────────────────────────────────────────────────────┘
""")